# Chapter 9: Geospatial and temporal health data

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. Time series in health

A *time series* is a sequence of measurements at regular intervals: daily case counts, weekly deaths, monthly vaccination doses.

Three components:
- **Trend.** Long-term direction --- is incidence rising or falling?
- **Seasonality.** Regular cycles --- malaria peaks during rainy seasons, influenza in winter.
- **Noise.** Random day-to-day fluctuation.

## 2. Plotting time series with pandas

### Date parsing and indexing

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

# Example: daily malaria cases at a district hospital
dates = pd.date_range("2023-01-01", periods=365, freq="D")
# Simulate seasonal pattern: peak during rainy season (June-September)
day_of_year = dates.dayofyear
seasonal = 50 + 40 * np.sin(2 * np.pi * (day_of_year - 90) / 365)
cases = np.maximum(0, seasonal + np.random.normal(0, 12, 365)).astype(int)

df = pd.DataFrame({"date": dates, "cases": cases})
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df.head()

### Rolling averages

Day-to-day fluctuation makes trends hard to see. A *rolling average* smooths the curve.

In [ ]:
df["cases_7d"] = df["cases"].rolling(window=7).mean()
df["cases_14d"] = df["cases"].rolling(window=14).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df.index, df["cases"], alpha=0.3, label="Daily cases", color="steelblue")
ax.plot(df.index, df["cases_7d"], color="red", linewidth=2, label="7-day average")
ax.plot(df.index, df["cases_14d"], color="darkgreen", linewidth=2, label="14-day average")
ax.set_xlabel("Date")
ax.set_ylabel("Malaria cases")
ax.set_title("Daily malaria cases with rolling averages")
ax.legend()
plt.tight_layout()
plt.show()

### Trend decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(df["cases"], model="additive", period=30)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
result.observed.plot(ax=axes[0], title="Observed")
result.trend.plot(ax=axes[1], title="Trend")
result.seasonal.plot(ax=axes[2], title="Seasonal")
result.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

## 3. COVID-19 case curves

### Loading JHU data

In [ ]:
# JHU CSSE COVID-19 confirmed cases (global, time series)
url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
       "master/csse_covid_19_data/csse_covid_19_time_series/"
       "time_series_covid19_confirmed_global.csv")
confirmed = pd.read_csv(url)
print(f"Shape: {confirmed.shape}")
confirmed.head()

### Reshaping wide to long format

In [ ]:
# Focus on 5 African countries
countries = ["South Africa", "Nigeria", "Kenya", "Egypt", "Morocco"]
africa = confirmed[confirmed["Country/Region"].isin(countries)]

# Group by country (some countries have province-level rows)
africa = africa.groupby("Country/Region").sum(numeric_only=True)
africa = africa.drop(columns=["Lat", "Long"], errors="ignore")

# Reshape: wide -> long
africa_long = africa.T
africa_long.index = pd.to_datetime(africa_long.index)
africa_long.index.name = "date"
africa_long.head()

### Computing daily cases and 7-day averages

In [ ]:
# Cumulative -> daily new cases
daily = africa_long.diff().clip(lower=0)  # clip removes negative corrections

# 7-day rolling average
daily_7d = daily.rolling(7).mean()

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
for country in countries:
    ax.plot(daily_7d.index, daily_7d[country], linewidth=2, label=country)
ax.set_xlabel("Date")
ax.set_ylabel("Daily new cases (7-day average)")
ax.set_title("COVID-19 daily cases in 5 African countries")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Introduction to geospatial data

Geospatial data links measurements to locations on Earth.
- **Shapefiles** (.shp) --- traditional GIS format
- **GeoJSON** (.geojson) --- single JSON file, easier to share

## 5. geopandas for disease mapping

In [ ]:
# !pip install geopandas mapclassify
import geopandas as gpd

# Natural Earth dataset (built into geopandas)
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
print(f"Columns: {world.columns.tolist()}")
print(f"CRS: {world.crs}")  # Coordinate Reference System
world.head()

### Choropleth maps

A *choropleth* map colors regions according to a variable --- the standard for mapping disease burden.

In [ ]:
# Filter to Africa
africa_map = world[world["continent"] == "Africa"].copy()

# Plot population estimate
fig, ax = plt.subplots(figsize=(10, 10))
africa_map.plot(column="pop_est", cmap="YlOrRd", legend=True,
                legend_kwds={"label": "Population", "shrink": 0.6},
                edgecolor="black", linewidth=0.5, ax=ax)
ax.set_title("Population estimates in African countries")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6. Mapping malaria prevalence by African country

In [ ]:
# Simulated malaria incidence data for mapping
# (Using simulated data as WHO API may be unavailable)
np.random.seed(42)
africa_countries = africa_map["name"].tolist()
malaria_data = pd.DataFrame({
    "country": africa_countries,
    "incidence_per_1000": np.random.exponential(80, len(africa_countries)).clip(0, 400)
})

# Set low values for North African countries
for c in ["Morocco", "Algeria", "Tunisia", "Libya", "Egypt"]:
    malaria_data.loc[malaria_data["country"] == c, "incidence_per_1000"] = \
        np.random.uniform(0, 2)

africa_malaria = africa_map.merge(malaria_data, left_on="name",
                                   right_on="country", how="left")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
africa_malaria.plot(column="incidence_per_1000", cmap="YlOrRd",
                     legend=True, missing_kwds={"color": "lightgrey"},
                     legend_kwds={"label": "Incidence per 1,000",
                                  "shrink": 0.6},
                     edgecolor="black", linewidth=0.5, ax=ax)
ax.set_title("Malaria incidence in Africa (simulated data)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 7. Combining temporal and spatial analysis

### Small multiples: one map per time period

In [ ]:
# Example: COVID-19 cumulative cases per country at 3 time points
dates_to_plot = ["2020-06-01", "2021-01-01", "2021-06-01"]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, date_str in zip(axes, dates_to_plot):
    date_col = pd.to_datetime(date_str).strftime("%-m/%-d/%y")
    if date_col in africa.columns:
        data_col = africa[date_col]
    else:
        # Find nearest date
        all_dates = pd.to_datetime(africa.columns, errors="coerce")
        nearest = all_dates[all_dates <= pd.to_datetime(date_str)][-1]
        data_col = africa[nearest.strftime("%-m/%-d/%y")]

    temp = africa_map.copy()
    temp = temp.merge(data_col.reset_index(), left_on="name",
                       right_on="Country/Region", how="left")
    temp.plot(column=data_col.name, cmap="Reds", legend=True,
              edgecolor="black", linewidth=0.5, ax=ax,
              missing_kwds={"color": "lightgrey"})
    ax.set_title(date_str)
    ax.set_axis_off()

plt.suptitle("COVID-19 cumulative cases in Africa", fontsize=14)
plt.tight_layout()
plt.show()

### Animated maps (optional advanced)

For presentations, animated GIFs can show epidemic spread over time.

In [ ]:
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(10, 10))

def update(frame_date):
    ax.clear()
    # Merge case data for this date with the map
    temp = africa_map.copy()
    # ... merge logic similar to above ...
    temp.plot(column="cases", cmap="Reds", ax=ax, edgecolor="black",
              linewidth=0.5, missing_kwds={"color": "lightgrey"},
              vmin=0, vmax=1000000)
    ax.set_title(f"COVID-19 cases - {frame_date}")
    ax.set_axis_off()

# Create animation (one frame per month)
# Uncomment to run:
# monthly_dates = pd.date_range("2020-03-01", "2021-06-01", freq="MS")
# ani = animation.FuncAnimation(fig, update, frames=monthly_dates, interval=500)
# ani.save("covid_africa.gif", writer="pillow")
print("Animation code ready (uncomment to generate GIF)")
plt.close()

## Exercises

Build a multi-panel COVID-19 dashboard for South Africa, Nigeria, Kenya, Egypt, and Morocco:

1. **Panel 1 - Confirmed cases over time.** Load JHU data, compute daily new cases and 7-day rolling averages, plot all 5 countries.
2. **Panel 2 - Deaths over time.** Load JHU deaths time series, compute daily deaths (7-day average).
3. **Panel 3 - Vaccination progress.** Load OWID vaccination data (`https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv`). Plot `people_fully_vaccinated_per_hundred`.
4. **Panel 4 - Choropleth map.** Create a choropleth of total confirmed cases across all African countries.
5. **Bonus:** Add a summary table with total cases, deaths, CFR, and vaccination rate per country.

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5 (bonus): your code here
